In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🔐 Secrets Management & Rotation Guide

**Secure secrets lifecycle from generation to rotation and revocation**

This guide covers:
- Secret generation strategies
- Secure storage backends
- Secret rotation policies
- Audit logging
- Revocation handling
- Secret version management
- Access control patterns
- Emergency procedures
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Secrets Management Architecture

### Secret Vault
```python
from dataclasses import dataclass, field
from typing import Dict, Optional, List
from datetime import datetime, timedelta
from enum import Enum
import hashlib

class SecretType(Enum):
    API_KEY = "api_key"
    PASSWORD = "password"
    JWT = "jwt"
    CERTIFICATE = "certificate"
    DATABASE_CREDENTIAL = "database_credential"

@dataclass
class SecretMetadata:
    """Secret metadata"""
    secret_id: str
    secret_type: SecretType
    owner: str
    created_at: datetime
    last_accessed: Optional[datetime] = None
    last_rotated: Optional[datetime] = None
    rotation_interval_days: int = 90
    version: int = 1
    is_active: bool = True

@dataclass
class SecretAuditLog:
    """Audit log entry"""
    timestamp: datetime
    secret_id: str
    action: str  # create, read, rotate, revoke
    actor: str
    status: str  # success, failure
    reason: Optional[str] = None

class SecretsVault:
    """Manage secrets securely"""
    
    def __init__(self):
        self.secrets: Dict[str, tuple] = {}  # secret_id -> (encrypted_value, metadata)
        self.secret_versions: Dict[str, List] = {}  # secret_id -> [versions]
        self.audit_logs: List[SecretAuditLog] = []
        self.access_policies: Dict[str, Dict] = {}
    
    def create_secret(self, secret_id: str, secret_value: str, 
                     secret_type: SecretType, owner: str) -> bool:
        """Create new secret"""
        
        # Check if secret already exists
        if secret_id in self.secrets:
            return False
        
        # Encrypt secret
        encrypted = self._encrypt_secret(secret_value)
        
        # Create metadata
        metadata = SecretMetadata(
            secret_id=secret_id,
            secret_type=secret_type,
            owner=owner,
            created_at=datetime.utcnow()
        )
        
        # Store secret
        self.secrets[secret_id] = (encrypted, metadata)
        self.secret_versions[secret_id] = [
            {
                'version': 1,
                'encrypted_value': encrypted,
                'created_at': datetime.utcnow().isoformat()
            }
        ]
        
        # Audit log
        self._log_audit('create', secret_id, owner, 'success')
        
        return True
    
    def get_secret(self, secret_id: str, accessor: str) -> Optional[str]:
        """Retrieve secret"""
        
        if secret_id not in self.secrets:
            self._log_audit('read', secret_id, accessor, 'failure', 'Not found')
            return None
        
        # Check access control
        if not self._check_access(secret_id, accessor, 'read'):
            self._log_audit('read', secret_id, accessor, 'failure', 'Access denied')
            return None
        
        encrypted, metadata = self.secrets[secret_id]
        
        # Decrypt secret
        decrypted = self._decrypt_secret(encrypted)
        
        # Update last accessed
        metadata.last_accessed = datetime.utcnow()
        
        # Audit log
        self._log_audit('read', secret_id, accessor, 'success')
        
        return decrypted
    
    def rotate_secret(self, secret_id: str, new_value: str, initiator: str) -> bool:
        """Rotate secret"""
        
        if secret_id not in self.secrets:
            self._log_audit('rotate', secret_id, initiator, 'failure', 'Not found')
            return False
        
        encrypted, metadata = self.secrets[secret_id]
        
        # Encrypt new value
        new_encrypted = self._encrypt_secret(new_value)
        
        # Update secret
        self.secrets[secret_id] = (new_encrypted, metadata)
        
        # Update version
        metadata.version += 1
        metadata.last_rotated = datetime.utcnow()
        
        # Add to versions
        self.secret_versions[secret_id].append({
            'version': metadata.version,
            'encrypted_value': new_encrypted,
            'created_at': datetime.utcnow().isoformat()
        })
        
        # Audit log
        self._log_audit('rotate', secret_id, initiator, 'success')
        
        return True
    
    def revoke_secret(self, secret_id: str, revoker: str, reason: str) -> bool:
        """Revoke secret"""
        
        if secret_id not in self.secrets:
            return False
        
        encrypted, metadata = self.secrets[secret_id]
        metadata.is_active = False
        
        # Audit log
        self._log_audit('revoke', secret_id, revoker, 'success', reason)
        
        return True
    
    def check_rotation_needed(self) -> List[str]:
        """Find secrets that need rotation"""
        
        secrets_to_rotate = []
        now = datetime.utcnow()
        
        for secret_id, (_, metadata) in self.secrets.items():
            if not metadata.is_active:
                continue
            
            last_rotated = metadata.last_rotated or metadata.created_at
            days_since_rotation = (now - last_rotated).days
            
            if days_since_rotation >= metadata.rotation_interval_days:
                secrets_to_rotate.append(secret_id)
        
        return secrets_to_rotate
    
    def _encrypt_secret(self, value: str) -> str:
        """Encrypt secret"""
        # Simplified - in production use proper encryption
        return hashlib.sha256(value.encode()).hexdigest()
    
    def _decrypt_secret(self, encrypted: str) -> str:
        """Decrypt secret"""
        # Simplified - in production use proper decryption
        return encrypted
    
    def _check_access(self, secret_id: str, actor: str, action: str) -> bool:
        """Check access control"""
        
        if secret_id not in self.access_policies:
            return False
        
        policy = self.access_policies[secret_id]
        allowed_actors = policy.get(action, [])
        
        return actor in allowed_actors
    
    def _log_audit(self, action: str, secret_id: str, actor: str, 
                   status: str, reason: Optional[str] = None):
        """Log audit entry"""
        
        log_entry = SecretAuditLog(
            timestamp=datetime.utcnow(),
            secret_id=secret_id,
            action=action,
            actor=actor,
            status=status,
            reason=reason
        )
        
        self.audit_logs.append(log_entry)
```

### Access Control
```python
class SecretAccessPolicy:
    """Manage secret access policies"""
    
    def __init__(self):
        self.policies: Dict[str, Dict] = {}
    
    def create_policy(self, secret_id: str, readers: List[str], 
                     rotators: List[str], revokers: List[str]):
        """Create access policy"""
        
        self.policies[secret_id] = {
            'read': readers,
            'rotate': rotators,
            'revoke': revokers
        }
    
    def grant_access(self, secret_id: str, actor: str, action: str) -> bool:
        """Grant access to secret"""
        
        if secret_id not in self.policies:
            return False
        
        if action not in self.policies[secret_id]:
            return False
        
        if actor not in self.policies[secret_id][action]:
            self.policies[secret_id][action].append(actor)
        
        return True
    
    def revoke_access(self, secret_id: str, actor: str, action: str) -> bool:
        """Revoke access to secret"""
        
        if secret_id not in self.policies:
            return False
        
        if action not in self.policies[secret_id]:
            return False
        
        if actor in self.policies[secret_id][action]:
            self.policies[secret_id][action].remove(actor)
        
        return True
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Rotation Strategies

### Automated Rotation
```python
class RotationScheduler:
    """Schedule and execute secret rotation"""
    
    def __init__(self):
        self.rotation_schedules: Dict[str, Dict] = {}
        self.rotation_history: List[Dict] = []
    
    def schedule_rotation(self, secret_id: str, interval_days: int, 
                         strategy: str = "generate_new"):
        """Schedule rotation"""
        
        self.rotation_schedules[secret_id] = {
            'interval_days': interval_days,
            'strategy': strategy,
            'last_rotation': None,
            'next_rotation': None
        }
    
    def execute_rotation(self, secret_id: str, vault: SecretsVault) -> bool:
        """Execute rotation"""
        
        if secret_id not in self.rotation_schedules:
            return False
        
        schedule = self.rotation_schedules[secret_id]
        
        # Generate new secret
        new_value = self._generate_new_secret(secret_id, schedule['strategy'])
        
        # Rotate in vault
        result = vault.rotate_secret(secret_id, new_value, 'rotation_scheduler')
        
        if result:
            schedule['last_rotation'] = datetime.utcnow()
            schedule['next_rotation'] = datetime.utcnow() + timedelta(days=schedule['interval_days'])
            
            self.rotation_history.append({
                'secret_id': secret_id,
                'timestamp': datetime.utcnow().isoformat(),
                'status': 'success'
            })
        
        return result
    
    def _generate_new_secret(self, secret_id: str, strategy: str) -> str:
        """Generate new secret"""
        
        import secrets
        import string
        
        if strategy == "generate_new":
            # Generate random string
            alphabet = string.ascii_letters + string.digits
            return ''.join(secrets.choice(alphabet) for _ in range(32))
        
        return ""
```

### Blue-Green Secret Rotation
```python
class BlueGreenSecretRotation:
    """Rotate secrets with zero downtime"""
    
    def __init__(self, vault: SecretsVault):
        self.vault = vault
        self.rotation_state: Dict[str, Dict] = {}
    
    def start_rotation(self, secret_id: str) -> bool:
        """Start rotation (green phase)"""
        
        # Create green version
        new_value = self._generate_new_secret()
        
        self.rotation_state[secret_id] = {
            'blue_version': self.vault.secrets[secret_id][1].version,
            'green_version': None,
            'state': 'generating_green',
            'created_at': datetime.utcnow()
        }
        
        # Update to new secret
        result = self.vault.rotate_secret(secret_id, new_value, 'rotation')
        
        if result:
            self.rotation_state[secret_id]['green_version'] = \
                self.vault.secrets[secret_id][1].version
            self.rotation_state[secret_id]['state'] = 'validating'
        
        return result
    
    def validate_and_promote(self, secret_id: str) -> bool:
        """Validate green and promote to blue"""
        
        if secret_id not in self.rotation_state:
            return False
        
        # After validation, mark new version as active
        self.rotation_state[secret_id]['state'] = 'promoted'
        
        return True
    
    def _generate_new_secret(self) -> str:
        """Generate new secret"""
        import secrets
        return secrets.token_hex(16)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Emergency Procedures

### Emergency Revocation
```python
class EmergencyRevocation:
    """Handle emergency secret revocation"""
    
    def __init__(self, vault: SecretsVault):
        self.vault = vault
        self.emergency_events: List[Dict] = []
    
    def emergency_revoke_all(self, reason: str, revoker: str) -> int:
        """Revoke all secrets"""
        
        revoked_count = 0
        
        for secret_id in list(self.vault.secrets.keys()):
            if self.vault.revoke_secret(secret_id, revoker, reason):
                revoked_count += 1
        
        self.emergency_events.append({
            'timestamp': datetime.utcnow().isoformat(),
            'action': 'emergency_revoke_all',
            'revoker': revoker,
            'reason': reason,
            'revoked_count': revoked_count
        })
        
        return revoked_count
    
    def emergency_revoke_by_type(self, secret_type: SecretType, 
                                reason: str, revoker: str) -> int:
        """Revoke secrets by type"""
        
        revoked_count = 0
        
        for secret_id, (_, metadata) in self.vault.secrets.items():
            if metadata.secret_type == secret_type:
                if self.vault.revoke_secret(secret_id, revoker, reason):
                    revoked_count += 1
        
        return revoked_count
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Secrets Management Checklist

✅ **Vault Setup**
- [ ] Secret vault implemented
- [ ] Encryption configured
- [ ] Storage backend secured
- [ ] Access control list
- [ ] Audit logging enabled

✅ **Rotation**
- [ ] Rotation policies defined
- [ ] Automated scheduler
- [ ] Blue-green rotation
- [ ] Validation gates
- [ ] Rollback capability

✅ **Access Control**
- [ ] Access policies enforced
- [ ] Role-based access
- [ ] Time-limited access
- [ ] Audit trail complete
- [ ] Revocation tracking

✅ **Emergency**
- [ ] Emergency procedures documented
- [ ] Mass revocation capability
- [ ] Incident response plan
- [ ] Communication templates
- [ ] Recovery procedures

✅ **Operational**
- [ ] Monitoring configured
- [ ] Alerts set up
- [ ] Documentation
- [ ] Team training
- [ ] Regular audits
</VSCode.Cell>
```